## 📑 Core NPE NPL Management Actual Results
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/JACeGw)**  
  Comprehensive documentation outlining the business requirements for this report.

#####Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
import boto3
import os
 
#takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
# env = os.environ["S3PREFIX"]

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

###### Schema of the table

In [0]:
table_name = "core_NPENPLManagementActualResults"

schema = (
         "BusinessDate date,"				
         "ExposureNumber string,"			
         "NPEAmountStartOfPeriod decimal(19,2),"	
         "NPLAmountStartOfPeriod decimal(19,2),"		
         "PEInflow decimal(19,2),"						
         "PLInflow decimal(19,2),"						
         "CFPLInflow decimal(19,2),"						
         "FPEInflow decimal(19,2),"					
         "FPLInflow decimal(19,2),"						
         "FNPLInflow decimal(19,2),"						
         "NPLInflow decimal(19,2),"						
         "InterestNPE decimal(19,2),"						
         "InterestNPL decimal(19,2),"						
         "NaturalCures decimal(19,2),"					
         "ModificationCures decimal(19,2),"				
         "Restructures decimal(19,2),"					
         "NPEPayments decimal(19,2),"						
         "NPLPayments decimal(19,2),"						
         "NPEModificationHaircuts decimal(19,2),"			
         "NPLModificationHaircuts decimal(19,2),"			
         "NPELiquidations decimal(19,2),"					
         "NPLLiquidations decimal(19,2),"					
         "WriteOffs decimal(19,2),"					
         "AssetSales decimal(19,2),"						
         "NPEOtherInflowOutflow decimal(19,2),"			
         "NPLOtherInflowOutflow decimal(19,2),"			
         "NPEAmountEndOfPeriod decimal(19,2),"			
         "NPLAmountEndOfPeriod decimal(19,2),"			
         "Perimeter string,"						
         "ReferenceDateUnion date,"
         "InsertTimeStamp timestamp"
         )

In [0]:
import delta_table_creator, extract, transform
from delta_table_creator import create_unity_delta_table
from transform import delete_from_unity_table

partition_columns = ["BusinessDate"]

delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
       
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
 
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date, # type: ignore
)

##### Create Delta Tables with selected fields & Temporary Views for each delta table

###### UnionFile

In [0]:
CurrentUnionFile = spark.sql(
    f'''
SELECT distinct 'Core Retail' PortfolioName
              , ExposureNumber 
              , ExposureSegmentation
              , cast(REPLACE(IFRSGrossCarryingAmount, ',', '') AS DECIMAL(19,2))  IFRSGrossCarryingAmount
              , HFSFlag 
              , FPLUnderProbationFlag
              , SnapshotDate
              , EBAStatus 
              , NPLWithMaterialityFlag 
              , CASE WHEN EBAStatus = 'PL'                                THEN 'PL'
			         WHEN EBAStatus = 'FPL' AND FPLUnderProbationFlag = 0 THEN 'FPL'
			         WHEN EBAStatus = 'FPL' AND FPLUnderProbationFlag = 1 THEN 'CFPL'
			         WHEN EBAStatus = 'FNPL'                              THEN 'FNPL'
			         WHEN EBAStatus = 'NPL'                               THEN 'NPL'
			         END AS EBAStatusFinal
	         , CASE WHEN ExposureSegmentation > 3 AND (HFSFlag = 0 OR HFSFlag IS NULL) THEN 1 ELSE 0 END AS RetailPerimeterFlag
FROM `{env_prefix}-cdh`.externalfiles.extunionfile
WHERE SnapshotDate = LAST_DAY(date_format(date_add(MONTH,-1,{run_date!r}),'yyyy-MM-dd'))
AND cast(IngestionTimeStamp as date) = (Select Max(cast(IngestionTimeStamp as date)) from `{env_prefix}-cdh`.externalfiles.extunionfile where SnapshotDate = LAST_DAY(date_format(date_add(MONTH,-1,{run_date!r}),'yyyy-MM-dd')))

''')
CurrentUnionFile.createOrReplaceTempView("CurrentUnionFile")

current_month_last_date = spark.sql(f'''SELECT LAST_DAY(date_format(date_add(MONTH,-1,{run_date!r}),'yyyy-MM-dd')) ''').collect()[0][0]

# Check if the dataset is empty and print a message accordingly
if CurrentUnionFile.count() == 0:
    print(f"The required external file extunionfile dataset for date {current_month_last_date!r} is empty")

PreviousUnionFile = spark.sql(
    f'''
SELECT distinct 'Core Retail' PortfolioName
              , ExposureNumber 
              , ExposureSegmentation
              , cast(REPLACE(IFRSGrossCarryingAmount, ',', '') AS DECIMAL(19,2)) IFRSGrossCarryingAmount
              , HFSFlag 
              , FPLUnderProbationFlag
              , SnapshotDate
              , EBAStatus 
              , NPLWithMaterialityFlag 
              , CASE WHEN EBAStatus = 'PL'                                THEN 'PL'
			         WHEN EBAStatus = 'FPL' AND FPLUnderProbationFlag = 0 THEN 'FPL'
			         WHEN EBAStatus = 'FPL' AND FPLUnderProbationFlag = 1 THEN 'CFPL'
			         WHEN EBAStatus = 'FNPL'                              THEN 'FNPL'
			         WHEN EBAStatus = 'NPL'                               THEN 'NPL'
			         END AS EBAStatusFinal
	          , CASE WHEN ExposureSegmentation >3 AND (HFSFlag = 0 OR HFSFlag IS NULL) THEN 1 ELSE 0 END AS RetailPerimeterFlag
FROM `{env_prefix}-cdh`.externalfiles.extunionfile
WHERE LAST_DAY(SnapshotDate) = LAST_DAY(date_format(date_add(MONTH,-2,{run_date!r}),'yyyy-MM-dd')) AND ExposureSegmentation > 3 AND (HFSFlag = 0 OR HFSFlag IS NULL)
AND cast(IngestionTimeStamp as date) = (Select Max(cast(IngestionTimeStamp as date)) from `{env_prefix}-cdh`.externalfiles.extunionfile where SnapshotDate = LAST_DAY(date_format(date_add(MONTH,-2,{run_date!r}),'yyyy-MM-dd')))

''')
PreviousUnionFile.createOrReplaceTempView("PreviousUnionFile")


previous_month_last_date = spark.sql(f'''SELECT LAST_DAY(date_format(date_add(MONTH,-2,{run_date!r}),'yyyy-MM-dd')) ''').collect()[0][0]

# Check if the dataset is empty and print a message accordingly
if PreviousUnionFile.count() == 0:
    print(f"The required external file extunionfile dataset for date {previous_month_last_date!r} is empty")

###### PaymentsMtD

In [0]:
PaymentsMtD = spark.sql(
    f'''
SELECT ExposureKey
     , SUM(PaymentAmount) AS PaymentAmount
FROM `{env_prefix}-cdh-delivery`.repoint.core_paymentsmtd
WHERE LAST_DAY(BusinessDate) = LAST_DAY(date_format(date_add(MONTH,-1,{run_date!r}),'yyyy-MM-dd')) 
GROUP BY ExposureKey
''')
PaymentsMtD.createOrReplaceTempView("PaymentsMtD")

###### Depreciations

In [0]:
depreciation = spark.sql(
    f'''
SELECT BusinessDate
     , ReportedMonth
     , AssetSegment
     , CDI
     , ExposureKey
     , cast(REPLACE(REPLACE(REPLACE(Amortization, '-',0),'.', ''),',', '.') AS DECIMAL(19,2)) Amortization
     , Category
     , CoreNoCoreFlag
FROM  `{env_prefix}-cdh`.externalfiles.extamortization
WHERE ReportedMonth = date_format(date_add(MONTH,-1,{run_date!r}),'yyyyMM') 
 AND (Category IN ('Clean Ups', 'Auctions') OR Category LIKE '%N3869%')
 AND IngestionTimeStamp = (Select Max(IngestionTimeStamp) from `{env_prefix}-cdh`.externalfiles.extamortization where ReportedMonth = date_format(date_add(MONTH,-1,{run_date!r}),'yyyyMM')) 
''')
depreciation.createOrReplaceTempView("depreciation")

ReportedMonth = spark.sql(f'''SELECT date_format(date_add(MONTH,-1,{run_date!r}),'yyyyMM') ''').collect()[0][0]

# Check if the dataset is empty and print a message accordingly
if depreciation.count() == 0:
    print(f"The required external file extamortization dataset for month {ReportedMonth!r} is empty")

###### NFPAYMENTS

In [0]:
NFPAYMENTS_Step0 = spark.sql(
    f'''
SELECT ACCOUNT_NUMBER
		 , PAYMENT_AMOUNT
     , RANK() OVER(PARTITION BY ACCOUNT_NUMBER,PAYMENT_AMOUNT,PAY_DATE ORDER BY BusinessDate DESC)  AS Rank
FROM `{env_prefix}-cdh-delivery`.repoint.nfpayments 
WHERE PAY_DATE > LAST_DAY(date_format(date_add(MONTH,-2,{run_date!r}),'yyyy-MM-dd')) AND PAY_DATE <= LAST_DAY(date_format(date_add(MONTH,-1,{run_date!r}),'yyyy-MM-dd')) AND PAYMENT_RESON_CODE IN ('84','85','90')
''')
NFPAYMENTS_Step0.createOrReplaceTempView("NFPAYMENTS_Step0")	

NFPAYMENTS = spark.sql(
    f'''
SELECT ACCOUNT_NUMBER
     , SUM(PAYMENT_AMOUNT) AS PAYMENT_AMOUNT
FROM NFPAYMENTS_Step0
WHERE Rank=1
GROUP BY ACCOUNT_NUMBER
''')
NFPAYMENTS.createOrReplaceTempView("NFPAYMENTS")

######Modifications

In [0]:
Modifications_Step0 = spark.sql(
    f'''
SELECT ModificationDate AS ModificationDate
     , OldExposureKey AS ParentAccount  
     , NewExposureKey AS ChildAccount  
     , AssetSegment As BU
     , CAST(HaircutAmount AS DECIMAL(19,2))   AS HaircutAmount
     , Category
	 , RANK() OVER(PARTITION BY ModificationDate,OldExposureKey,NewExposureKey ORDER BY BusinessDate DESC)  AS Rank
FROM `{env_prefix}-cdh-delivery`.repoint.core_modifications
WHERE Month = date_format(date_add(MONTH,-1,{run_date!r}),'yyyyMM')
''')
Modifications_Step0.createOrReplaceTempView("Modifications_Step0")

Modifications_Step1 = spark.sql(
    f'''
SELECT ModificationDate  
     , ParentAccount   
     , ChildAccount  
     , BU
     , HaircutAmount
     , Category
FROM Modifications_Step0
WHERE Rank=1
''')
Modifications_Step1.createOrReplaceTempView("Modifications_Step1")


Modifications_Haircut = spark.sql(
    f'''
SELECT DISTINCT ParentAccount
               ,IFNULL(HaircutAmount,0) AS HaircutAmount
FROM Modifications_Step1
WHERE Category NOT LIKE '%3869%'
''')
Modifications_Haircut.createOrReplaceTempView("Modifications_Haircut")


Modifications_Step2 = spark.sql(
    f'''
SELECT ModificationDate 
     , ParentAccount
     , ChildAccount
     , HaircutAmount 
     , Category 
	 , u.EBAStatusFinal AS EBAStatus
	 , u.NPLWithMaterialityFlag
	 , u.IFRSGrossCarryingAmount
	 , CASE WHEN u.EBAStatusFinal = 'PL'   THEN 0
		    WHEN u.EBAStatusFinal = 'FPL'  THEN 1
		    WHEN u.EBAStatusFinal = 'CFPL' THEN 2
		    WHEN u.EBAStatusFinal = 'NPL'  THEN 3
		    WHEN u.EBAStatusFinal = 'FNPL' THEN 4
			END AS Priority
FROM Modifications_Step1     m
INNER JOIN PreviousUnionFile u
ON u.ExposureNumber = m.ParentAccount
''')
Modifications_Step2.createOrReplaceTempView("Modifications_Step2")  


Modifications_DistinctChild = spark.sql(
    f'''
SELECT DISTINCT ChildAccount
FROM Modifications_Step2  
''')
Modifications_DistinctChild.createOrReplaceTempView("Modifications_DistinctChild")  


Modifications_Step3 = spark.sql(
    f'''
SELECT  ModificationDate 
      , ParentAccount
      , ChildAccount
      , HaircutAmount 
      , Category 
	  , EBAStatus
      , Priority
	  , NPLWithMaterialityFlag
	  , IFNULL(CAST(IFRSGrossCarryingAmount AS DECIMAL(19,2)),0) AS IFRSGrossCarryingAmount
	  , RANK() OVER(PARTITION BY ChildAccount ORDER BY Priority ASC) AS Rank
	  , RANK() OVER(PARTITION BY ChildAccount ORDER BY NPLWithMaterialityFlag ASC) AS Rank1  
FROM Modifications_Step2
''')
Modifications_Step3.createOrReplaceTempView("Modifications_Step3")   

Modifications = spark.sql(
    f'''
SELECT ModificationDate 
     , ParentAccount
     , ChildAccount
     , HaircutAmount 
     , Category 
	 , EBAStatus
	 , Priority
	 , NPLWithMaterialityFlag
FROM Modifications_Step3
WHERE Rank = 1 AND Rank1 = 1
''')
Modifications.createOrReplaceTempView("Modifications")   

ModificationsIFRSGrossCarryingAmt = spark.sql(
    f'''
SELECT ChildAccount
     , SUM(IFRSGrossCarryingAmount) AS IFRSGrossCarryingAmount
FROM Modifications_Step3
GROUP BY ChildAccount
''')
ModificationsIFRSGrossCarryingAmt.createOrReplaceTempView("ModificationsIFRSGrossCarryingAmt")     

###### Union Perimeter

In [0]:
Union_Perimeter1 = spark.sql(
    f'''
SELECT c.* 
	, 'Type 1' AS Perimeter
FROM CurrentUnionFile        c
LEFT JOIN PreviousUnionFile  p   
ON p.ExposureNumber = c.ExposureNumber    
WHERE c.RetailPerimeterFlag = 1
''')
Union_Perimeter1.createOrReplaceTempView("Union_Perimeter1")   

Union_PerimeterModified = spark.sql(
    f'''
SELECT c.* 
	 , 'Type 1' AS Perimeter
FROM CurrentUnionFile                  c
INNER JOIN Modifications_DistinctChild p
ON p.ChildAccount = c.ExposureNumber    
WHERE c.RetailPerimeterFlag = 1
''')
Union_PerimeterModified.createOrReplaceTempView("Union_PerimeterModified")  

Union_Perimeter2 = spark.sql(
    f'''
SELECT *
	 , 'Type 2' AS Perimeter
FROM PreviousUnionFile     
WHERE EBAStatus IN ('FNPL','NPL')
''')
Union_Perimeter2.createOrReplaceTempView("Union_Perimeter2") 

Union_Perimeter3 = spark.sql(
    f'''
SELECT *
	, 'Type 3' AS Perimeter
FROM PreviousUnionFile     
WHERE NPLWithMaterialityFlag = 1  
''')
Union_Perimeter3.createOrReplaceTempView("Union_Perimeter3")    

Union = spark.sql(
    f'''
SELECT * FROM Union_Perimeter1      
		
UNION 

SELECT * FROM Union_PerimeterModified

UNION 
		
SELECT * FROM Union_Perimeter2 

UNION

SELECT * FROM Union_Perimeter3 
''')
Union.createOrReplaceTempView("Union")   

In [0]:
Union_Step1 = spark.sql(
    f'''
SELECT  u.ExposureNumber
	  , u.EBAStatusFinal
	  , u.Perimeter
	  , u.NPLWithMaterialityFlag
	  , CASE WHEN p.ExposureNumber IS NOT NULL        THEN p.EBAStatusFinal
			 WHEN child.ChildAccount IS NOT NULL  THEN child.EBAStatus 
			 WHEN c.ExposureNumber IS NOT NULL        THEN c.EBAStatusFinal
			 END AS PreviousEBAStatus
      , CASE WHEN p.ExposureNumber IS NOT NULL        THEN p.NPLWithMaterialityFlag
			 WHEN child.ChildAccount IS NOT NULL  THEN child.NPLWithMaterialityFlag 
			 WHEN c.ExposureNumber IS NOT NULL        THEN c.NPLWithMaterialityFlag
			 END AS PreviousNPLWithMaterialityFlag
	  , CASE  WHEN p.ExposureNumber IS NOT NULL    THEN IFNULL(p.IFRSGrossCarryingAmount,0)
			  WHEN i.ChildAccount IS NOT NULL  THEN IFNULL(i.IFRSGrossCarryingAmount,0)
			  WHEN c.ExposureNumber IS NOT NULL    THEN IFNULL(c.IFRSGrossCarryingAmount,0)
			  ELSE 0 END AS PreviousIFRSGrossCarryingAmount
	  , CASE WHEN Perimeter IN ('Type 2','Type 3') AND parent.ParentAccount IS NOT NULL THEN IFNULL(parent.HaircutAmount,0)
			 ELSE 0 END AS HaircutAmount
	  , CASE WHEN Perimeter IN ('Type 2','Type 3') AND a.ExposureKey IS NOT NULL AND a.Category LIKE '%N3869%' THEN IFNULL(Amortization,0)
			 ELSE 0 END AS HaircutAmountL3869
      , CASE WHEN Perimeter IN ('Type 2','Type 3') THEN n.PAYMENT_AMOUNT END AS LiqPayments   
      , CASE WHEN Perimeter IN ('Type 2','Type 3') AND a.ExposureKey IS NOT NULL AND a.Category = 'Auctions' THEN IFNULL(Amortization,0)
			 ELSE 0 END AS LiqWOFFS
      , CASE WHEN Perimeter IN ('Type 2','Type 3') AND a.ExposureKey IS NOT NULL AND a.Category = 'Clean Ups' THEN IFNULL(Amortization,0)
			 ELSE 0 END AS WriteOffs
      , CASE WHEN Perimeter IN ('Type 2','Type 3') AND c.EBAStatusFinal IN ('FNPL','NPL') AND c.HFSFlag=1 THEN 'Yes'
			 ELSE 'No' END AS AssetSales
FROM Union                                  u
LEFT JOIN Modifications                 child
ON child.ChildAccount = u.ExposureNumber
LEFT JOIN Modifications_Haircut        parent
ON parent.ParentAccount = u.ExposureNumber
LEFT JOIN ModificationsIFRSGrossCarryingAmt i
ON i.ChildAccount = u.ExposureNumber
LEFT JOIN CurrentUnionFile                  c
ON c.ExposureNumber = u.ExposureNumber
LEFT JOIN PreviousUnionFile                 p
ON p.ExposureNumber = u.ExposureNumber			     
LEFT JOIN depreciation                        a
ON a.ExposureKey = u.ExposureNumber		     
LEFT JOIN NFPAYMENTS                        n
ON n.ACCOUNT_NUMBER = u.ExposureNumber
''')
Union_Step1.createOrReplaceTempView("Union_Step1")     

In [0]:
Union_Step2 = spark.sql(
    f'''
		SELECT  u.*
		      , CASE WHEN Perimeter = 'Type 1' AND PreviousEBAStatus = 'PL' AND (u.EBAStatusFinal = 'FNPL' OR u.EBAStatusFinal = 'NPL')    THEN 'PE Inflow'
					 WHEN Perimeter = 'Type 1' AND PreviousEBAStatus = 'FPL' AND (u.EBAStatusFinal = 'FNPL' OR u.EBAStatusFinal = 'NPL')   THEN 'FPE Inflow' 
					 WHEN Perimeter = 'Type 1' AND PreviousEBAStatus = 'CFPL' AND (u.EBAStatusFinal = 'FNPL' OR u.EBAStatusFinal = 'NPL')  THEN 'CFPL Inflow'
					 WHEN Perimeter = 'Type 1' AND (PreviousEBAStatus = 'FNPL' OR PreviousEBAStatus = 'NPL') AND u.EBAStatusFinal = 'PL'   THEN 'Natural Cures'
					 WHEN Perimeter = 'Type 1' AND (PreviousEBAStatus = 'FNPL' OR PreviousEBAStatus = 'NPL') AND u.EBAStatusFinal = 'CFPL' THEN 'Modification Cures'
                END AS PEInflowCureStatus

			  , CASE WHEN Perimeter = 'Type 1' AND PreviousEBAStatus = 'PL' AND u.NPLWithMaterialityFlag = 1 AND u.PreviousNPLWithMaterialityFlag = 0                                  THEN 'PL Inflow'
					 WHEN Perimeter = 'Type 1' AND (PreviousEBAStatus = 'FPL' OR PreviousEBAStatus = 'CFPL') AND u.NPLWithMaterialityFlag = 1 AND u.PreviousNPLWithMaterialityFlag = 0 THEN 'FPL Inflow'
					 WHEN Perimeter = 'Type 1' AND PreviousEBAStatus = 'FNPL' AND u.NPLWithMaterialityFlag = 1 AND u.PreviousNPLWithMaterialityFlag = 0                                THEN 'FNPL Inflow'
					 WHEN Perimeter = 'Type 1' AND PreviousEBAStatus = 'NPL' AND u.NPLWithMaterialityFlag = 1 AND u.PreviousNPLWithMaterialityFlag = 0                                 THEN 'NPL Inflow'  
					 WHEN Perimeter = 'Type 1' AND c.EBAStatus = 'PL' AND c.NPLWithMaterialityFlag = 0 AND u.PreviousNPLWithMaterialityFlag = 1                                       THEN 'Natural Cures'   
                END AS PLInflowCureStatus

              , CASE WHEN Perimeter = 'Type 1' AND PreviousEBAStatus IN ('FNPL','NPL') AND c.EBAStatusFinal IN ('FNPL','NPL')      THEN IFNULL(PaymentAmount,0) ELSE 0 END  AS NPEPayments
              , CASE WHEN Perimeter = 'Type 1' AND u.PreviousNPLWithMaterialityFlag = 1 AND u.NPLWithMaterialityFlag = 1 THEN IFNULL(PaymentAmount,0) ELSE 0 END  AS NPLPayments
              , CASE WHEN Perimeter = 'Type 2' THEN IFNULL(HaircutAmount,0) + IFNULL(HaircutAmountL3869,0) ELSE 0 END   AS NPEModificationHaircuts
			  , CASE WHEN Perimeter = 'Type 3' THEN IFNULL(HaircutAmount,0) + IFNULL(HaircutAmountL3869,0) ELSE 0 END   AS NPLModificationHaircuts
			  , CASE WHEN Perimeter = 'Type 2' THEN IFNULL(LiqPayments,0) + IFNULL(LiqWOFFS,0) ELSE 0 END     AS NPELiquidations  
			  , CASE WHEN Perimeter = 'Type 3' THEN IFNULL(LiqPayments,0) + IFNULL(LiqWOFFS,0) ELSE 0 END     AS NPLLiquidations  
			  , CASE WHEN Perimeter = 'Type 1' AND c.RetailPerimeterFlag = 1
		              AND PreviousEBAStatus IN ('FNPL','NPL') AND c.EBAStatusFinal IN ('FNPL','NPL') AND IFNULL(c.IFRSGrossCarryingAmount,0) - IFNULL(u.PreviousIFRSGrossCarryingAmount,0) > 0
			         THEN IFNULL(c.IFRSGrossCarryingAmount,0) - IFNULL(u.PreviousIFRSGrossCarryingAmount,0) + IFNULL(pay.PaymentAmount,0) 
					 ELSE 0 
					 END AS InterestNPE    
			  , CASE WHEN Perimeter = 'Type 1' AND c.RetailPerimeterFlag = 1
		              AND u.PreviousNPLWithMaterialityFlag=1 AND c.NPLWithMaterialityFlag=1 AND IFNULL(c.IFRSGrossCarryingAmount,0) - IFNULL(u.PreviousIFRSGrossCarryingAmount,0) > 0
			         THEN IFNULL(c.IFRSGrossCarryingAmount,0) - IFNULL(u.PreviousIFRSGrossCarryingAmount,0) + IFNULL(pay.PaymentAmount,0)
					 ELSE 0 
					 END AS InterestNPL 
FROM Union_Step1            u
LEFT JOIN PaymentsMtD     pay
ON pay.ExposureKey = u.ExposureNumber
LEFT JOIN CurrentUnionFile  c
ON c.ExposureNumber = u.ExposureNumber
''')
Union_Step2.createOrReplaceTempView("Union_Step2")   

In [0]:
Final_Step1 = spark.sql(
    f'''
	    SELECT u.ExposureNumber
		      , CASE WHEN p.EBAStatusFinal IN ('FNPL','NPL') THEN IFNULL(u.PreviousIFRSGrossCarryingAmount,0) ELSE 0 END AS NPEAmountStartOfPeriod 
			  , CASE WHEN p.NPLWithMaterialityFlag = 1  THEN IFNULL(u.PreviousIFRSGrossCarryingAmount,0) ELSE 0 END AS NPLAmountStartOfPeriod    
	          , CASE WHEN PEInflowCureStatus =  'PE Inflow' THEN IFNULL(c.IFRSGrossCarryingAmount,0) ELSE 0 END          AS PEInflow
			  , CASE WHEN PLInflowCureStatus =  'PL Inflow' THEN IFNULL(c.IFRSGrossCarryingAmount,0) ELSE 0 END          AS PLInflow
			  , CASE WHEN PEInflowCureStatus =  'CFPL Inflow' THEN IFNULL(c.IFRSGrossCarryingAmount,0) ELSE 0 END        AS CFPLInflow   
			  , CASE WHEN PEInflowCureStatus =  'FPE Inflow' THEN IFNULL(c.IFRSGrossCarryingAmount,0) ELSE 0 END         AS FPEInflow   
			  , CASE WHEN PLInflowCureStatus =  'FPL Inflow' THEN IFNULL(c.IFRSGrossCarryingAmount,0) ELSE 0 END         AS FPLInflow
			  , CASE WHEN PLInflowCureStatus =  'FNPL Inflow'THEN IFNULL(c.IFRSGrossCarryingAmount,0) ELSE 0 END         AS FNPLInflow		
			  , CASE WHEN PLInflowCureStatus =  'NPL Inflow' THEN IFNULL(c.IFRSGrossCarryingAmount,0) ELSE 0 END         AS NPLInflow
			  , InterestNPE
			  , InterestNPL
			  , CASE WHEN PEInflowCureStatus =  'Natural Cures' THEN IFNULL(c.IFRSGrossCarryingAmount,0) ELSE 0 END      AS NaturalCures
			  , CASE WHEN PEInflowCureStatus =  'Modification Cures' THEN IFNULL(c.IFRSGrossCarryingAmount,0) ELSE 0 END AS ModificationCures
			  , CASE WHEN parent.ParentAccount IS NOT NULL AND u.PreviousNPLWithMaterialityFlag = 0 THEN IFNULL(u.PreviousIFRSGrossCarryingAmount,0)  
			         WHEN parent.ParentAccount IS NOT NULL AND u.PreviousNPLWithMaterialityFlag = 1 THEN IFNULL(u.PreviousIFRSGrossCarryingAmount - u.HaircutAmount,0)    
			         ELSE 0 END AS Restructures    
			  , NPEPayments
			  , NPLPayments
			  , NPEModificationHaircuts
			  , NPLModificationHaircuts
			  , NPELiquidations
			  , NPLLiquidations
			  , WriteOffs
			  , CASE WHEN AssetSales = 'Yes' THEN IFNULL(c.IFRSGrossCarryingAmount,0) ELSE 0 END                       AS AssetSales
			  , CASE WHEN u.EBAStatusFinal IN ('FNPL','NPL') THEN IFNULL(c.IFRSGrossCarryingAmount,0) ELSE 0 END         AS NPEAmountEndOfPeriod
			  , CASE WHEN u.NPLWithMaterialityFlag = 1 THEN IFNULL(c.IFRSGrossCarryingAmount,0) ELSE 0 END       AS NPLAmountEndOfPeriod
			  , Perimeter
              , CAST(LAST_DAY(date_format(date_add(MONTH,-1,{run_date!r}),'yyyy-MM-dd')) AS DATE) AS ReferenceDateUnion
FROM Union_Step2              u
LEFT JOIN CurrentUnionFile    c
ON c.ExposureNumber = u.ExposureNumber
LEFT JOIN PreviousUnionFile   p
ON p.ExposureNumber = u.ExposureNumber
LEFT JOIN Modifications_Step1 parent  
ON parent.ParentAccount = u.ExposureNumber
''')
Final_Step1.createOrReplaceTempView("Final_Step1")      

In [0]:
Final_Step2 = spark.sql(
    f'''
SELECT *
     , 1 Ranking
FROM Final_Step1
WHERE Perimeter='Type 1'

UNION

SELECT *
FROM (
SELECT *
     , RANK()OVER(PARTITION BY ExposureNumber ORDER BY Perimeter ASC) Ranking
FROM Final_Step1
WHERE Perimeter IN ('Type 2','Type 3')
)
WHERE Ranking=1
''')
Final_Step2.createOrReplaceTempView("Final_Step2")  

##### final_df

In [0]:
final_df = spark.sql(
    f'''
SELECT DISTINCT CAST({run_date!r}						 AS DATE)          AS BusinessDate				
               ,CAST(ExposureNumber						 AS STRING)        AS ExposureNumber				
               ,CAST(NPEAmountStartOfPeriod				 AS DECIMAL(19,2)) AS NPEAmountStartOfPeriod		
               ,CAST(NPLAmountStartOfPeriod				 AS DECIMAL(19,2)) AS NPLAmountStartOfPeriod		
               ,CAST(PEInflow							 AS DECIMAL(19,2)) AS PEInflow					
               ,CAST(PLInflow							 AS DECIMAL(19,2)) AS PLInflow					
               ,CAST(CFPLInflow							 AS DECIMAL(19,2)) AS CFPLInflow					
               ,CAST(FPEInflow							 AS DECIMAL(19,2)) AS FPEInflow					
               ,CAST(FPLInflow							 AS DECIMAL(19,2)) AS FPLInflow					
               ,CAST(FNPLInflow							 AS DECIMAL(19,2)) AS FNPLInflow					
               ,CAST(NPLInflow							 AS DECIMAL(19,2)) AS NPLInflow					
               ,CAST(InterestNPE						 AS DECIMAL(19,2)) AS InterestNPE					
               ,CAST(InterestNPL						 AS DECIMAL(19,2)) AS InterestNPL					
               ,CAST(NaturalCures						 AS DECIMAL(19,2)) AS NaturalCures				
               ,CAST(ModificationCures					 AS DECIMAL(19,2)) AS ModificationCures			
               ,CAST(Restructures						 AS DECIMAL(19,2)) AS Restructures				
               ,CAST(NPEPayments						 AS DECIMAL(19,2)) AS NPEPayments					
               ,CAST(NPLPayments						 AS DECIMAL(19,2)) AS NPLPayments					
               ,CAST(NPEModificationHaircuts			 AS DECIMAL(19,2)) AS NPEModificationHaircuts		
               ,CAST(NPLModificationHaircuts			 AS DECIMAL(19,2)) AS NPLModificationHaircuts		
               ,CAST(NPELiquidations					 AS DECIMAL(19,2)) AS NPELiquidations				
               ,CAST(NPLLiquidations					 AS DECIMAL(19,2)) AS NPLLiquidations				
               ,CAST(WriteOffs							 AS DECIMAL(19,2)) AS WriteOffs					
               ,CAST(AssetSales							 AS DECIMAL(19,2)) AS AssetSales					
               ,CAST(NULL				                 AS DECIMAL(19,2)) AS NPEOtherInflowOutflow		
               ,CAST(NULL				                 AS DECIMAL(19,2)) AS NPLOtherInflowOutflow		
               ,CAST(NPEAmountEndOfPeriod				 AS DECIMAL(19,2)) AS NPEAmountEndOfPeriod		
               ,CAST(NPLAmountEndOfPeriod				 AS DECIMAL(19,2)) AS NPLAmountEndOfPeriod		
               ,CAST(Perimeter							 AS STRING)        AS Perimeter					
               ,CAST(ReferenceDateUnion                  AS DATE)          AS ReferenceDateUnion	
               ,CAST(getdate()                           AS timestamp)     AS InsertTimeStamp
FROM Final_Step2
''')
final_df.createOrReplaceTempView("final_df")    

In [0]:
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_NPENPLManagementActualResults: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ExposureNumber", "Perimeter"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
npe_npl_manag_act_results = spark.sql(
    f'''
    select * from {delivery_table_path}
    where BusinessDate = '{run_date}'
    '''
)
npe_npl_manag_act_results.createOrReplaceTempView("npe_npl_manag_act_results")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/NPE_NPL_Management_Actual_Results/"
file_name = "NPE_NPL_Management_Actual_Results_" +run_date+ ".csv"
npe_npl_manag_act_results.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)